<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/05-data-sources-io/05-reading-from-kafka.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 5.5 — Reading from Kafka

No local Kafka needed: we build a DataFrame with **exactly the seven
columns and binary types the Kafka connector returns**, then run the
real deserialization pipeline on it. The `format("kafka")` code is
included (commented) — point it at any broker and it runs unchanged.

In [ ]:
import os, sys, json
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("5.5-kafka-bridge")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## The real thing (commented) — and our faithful fake

In [ ]:
# THE REAL READ — needs a broker + the connector package:
# spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0 ...
#
# raw = (spark.read.format("kafka")
#        .option("kafka.bootstrap.servers", "localhost:9092")
#        .option("subscribe", "orders")
#        .option("startingOffsets", "earliest")
#        .option("endingOffsets", "latest")
#        .load())

# THE FAKE: same 7 columns, same types (key/value BINARY), same semantics
msgs = [
    ("o-1001", {"order_id": 1001, "customer_id": "c001", "amount": 29.97}, 0, 100),
    ("o-1002", {"order_id": 1002, "customer_id": "c002", "amount": 18.50}, 0, 101),
    ("o-1003", {"order_id": 1003, "customer_id": "c003", "amount": 199.00}, 1, 87),
    ("o-1004", None,                                                        1, 88),   # tombstone/garbage
    ("o-1005", {"order_id": 1005, "customer_id": "c004", "amount": 24.00}, 2, 45),
]
rows = [
    (k.encode(), (json.dumps(v).encode() if v else b"not json at all"),
     "orders", p, o, None, 0)
    for k, v, p, o in msgs
]
raw = spark.createDataFrame(
    rows,
    "key BINARY, value BINARY, topic STRING, partition INT, offset LONG, "
    "timestamp TIMESTAMP, timestampType INT",
)
raw.printSchema()   # <- byte-for-byte what format("kafka") gives you
raw.show(truncate=40)

## Deserialize: cast + from_json with an explicit schema

In [ ]:
ORDER_EVENT = "order_id INT, customer_id STRING, amount DOUBLE"

events = raw.select(
    col("key").cast("string").alias("order_key"),
    F.from_json(col("value").cast("string"), ORDER_EVENT).alias("e"),
    "partition", "offset",
)
events.show(truncate=False)   # note the null 'e' for the garbage message

In [ ]:
# Split good rows from the dead-letter channel (Capstone 2's core move)
good = events.filter(col("e").isNotNull()).select("order_key", "e.*", "partition", "offset")
dead = events.filter(col("e").isNull())

good.show()
print("dead letters:", dead.count(), "-> in production: write these to a quarantine sink")

## Kafka partitions become Spark partitions — lineage columns prove it

In [ ]:
good.groupBy("partition").agg(
    F.count("*").alias("messages"),
    F.min("offset").alias("first_offset"),
    F.max("offset").alias("last_offset"),
).orderBy("partition").show()
# keep partition/offset through the pipeline: free provenance + dedup key

## The bridge to Module 14

The streaming version of everything above changes ~2 lines:

```python
raw = (spark.readStream.format("kafka")        # read -> readStream
       .option("kafka.bootstrap.servers", "...")
       .option("subscribe", "orders")
       .load())
# ...identical from_json pipeline...
query = (good.writeStream.format("parquet")     # write -> writeStream + checkpoint
         .option("path", "lake/orders")
         .option("checkpointLocation", "chk/orders")   # <- the offset memory
         .start())
```

The `checkpointLocation` is what the batch API deliberately lacks —
where processed offsets live, making exactly-once possible. That story
is Module 14.

## Exercises

1. Add a `ts TIMESTAMP` field to the event schema and messages; extract the *event* time and compare it to the Kafka *ingestion* timestamp column conceptually — why do both exist? (Module 14's watermark lesson depends on this distinction.)
2. Make the dead-letter channel real: write `dead` (with its raw `value` cast to string, plus partition/offset) to a local `quarantine/` parquet dir.
3. Simulate a duplicate delivery (same partition+offset appearing twice) and dedup with `dropDuplicates(["partition", "offset"])`. Why is THIS dedup key exactly right for Kafka (3.3's arbitrary-survivor caveat doesn't bite)?
4. If a 3-partition topic must feed a 24-core cluster with heavy per-row parsing, where does `repartition(24)` belong in the pipeline — before or after `from_json`? Reason about what each ordering shuffles.

In [ ]:
# your exercise code here
